# Environment Base Jax Friendly

> Base class for CRLD environments

In [ ]:
#| default_exp Environments/BaseJ

In [ ]:
#| hide
# Imports for the nbdev development environment
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| export
from fastcore.foundation import patch
from fastcore.utils import *
import jax.numpy as jnp
from jax import random
from collections.abc import Iterable


In [ ]:
#| export
class ebasej(object):
    """Base environment. All environments should inherit from this one."""
    
    def __init__(self):
        # Initialize a PRNG key for random sampling if not provided.
        self.rng = random.PRNGKey(0)
        
        self.T = self.TransitionTensor()
        self.F = jnp.array(self.FinalStates())
        self.R = self.RewardTensor()
        self.O = self.ObservationTensor()
                
        self.Aset = self.actions()
        self.Sset = self.states() 
        self.Oset = self.observations()

        # CHECKS
        R, T, O = self.R, self.T, self.O
        
        # number of agents
        N = R.shape[0]  
        assert O.shape[0] == N, "Inconsistent number of agents"
        assert len(T.shape[1:-1]) == N, "Inconsistent number of agents"
        assert len(R.shape[2:-1]) == N, "Inconsistent number of agents"
        
        # number of actions for each agent        
        M = T.shape[1] 
        assert all(x == M for x in T.shape[1:-1]), 'Inconsistent number of actions'
        assert all(x == M for x in R.shape[2:-1]), 'Inconsistent number of actions'
        assert all(len(a) == M for a in self.Aset), 'Inconsistent number of actions'
            
        # number of states
        Z = T.shape[0] 
        assert T.shape[-1] == Z, 'Inconsistent number of states'
        assert R.shape[-1] == Z, 'Inconsistent number of states'
        assert R.shape[1] == Z, 'Inconsistent number of states'
        assert O.shape[1] == Z, 'Inconsistent number of states'
        assert len(self.F) == Z, 'Inconsistent number of states'
        assert len(self.Sset) == Z, 'Inconsistent number of states'

        # number of observations
        Q = O.shape[-1]
        assert all(len(o) == Q for o in self.Oset), 'Inconsistent number of observations'
        
        assert jnp.allclose(T.sum(-1), 1), 'Transition model wrong'
        assert jnp.allclose(O.sum(-1), 1), 'Observation model wrong'


The `ebase` class `__init__` mostly contains consistency checks.

## Core methods

These need to be implemented by a concrete environment.

The transitions tensor `Tsjas'` gives the probability of the environment to transition to state `s'`, given that it was in state `s` and the agent chose the joint action `ja`.

In [ ]:
#| export
@patch
def TransitionTensor(self:ebasej):
        # Must be implemented by subclass
        raise NotImplementedError

In [ ]:
class slf: pass
test_fail(ebasej.TransitionTensor, args=slf)

raises `NotImplementedError`.

The reward tensor `Risjas'` gives the reward agent `i` receives when the environment is in state `s`, all agents choose the join action `ja`, and the environment transitions to state `s'`.

In [ ]:
#| export
@patch
def RewardTensor(self:ebasej):
        # Must be implemented by subclass
        raise NotImplementedError

In [ ]:
class slf: pass
test_fail(ebasej.RewardTensor, args=slf)

raises `NotImplementedError`.

The following two "core" methods are optional. If the concrete environment class does not implement them, they default to the following:

The observation tensor `Oiso` gives the probability that agent `i` observes observation `o` when the environment is in state `s`. The default observation tensor assumes perfect observation and sets the number of observations `Q` to the number of states `Z`.

In [ ]:
#| export
@patch
def ObservationTensor(self:ebasej):
        """Default observation tensor: perfect observation"""
        self.defaultObsTensUsed = True
        self.Q = self.Z
        # Create an identity matrix for each agent
        Oiso = jnp.stack([jnp.eye(self.Q) for _ in range(self.N)], axis=0)
        return Oiso

In [ ]:
class slf: Z = 2; N = 3  # dummy self for demonstration only
ebasej.ObservationTensor(slf)

Array([[[1., 0.],
        [0., 1.]],

       [[1., 0.],
        [0., 1.]],

       [[1., 0.],
        [0., 1.]]], dtype=float32)

Final states `Fs` indicate which states of the environment cause the end of an episode. Their meaning and use within CRLD are not fully resolved yet. If an environment does not implement `FinalStates` they default to no final states.

In [ ]:
#| export
@patch
def FinalStates(self:ebasej):
        """Default final states: no final states"""
        return jnp.zeros(self.Z, dtype=int)

In [ ]:
class slf: Z = 7 # dummy self for demonstration only
ebasej.FinalStates(slf)

Array([0, 0, 0, 0, 0, 0, 0], dtype=int32)

## Default string representations
String representations of actions, states and observations help with interpreting the results of simulation runs. Ideally, an environment class will implement these methods with descriptive values.

To show these methods here we create a dummy "self" of 2 environmental states, containing 3 agents with 4 actions and 5 observations of the environmental states.

In [ ]:
# dummy self of 2 environmental 2 agents with 3 actions in an environment
class slf: Z = 2; N = 3; M=4; Q=5

In [ ]:
#| export
@patch
def actions(self:ebasej):
        """Default action set representations `act_im`."""
        return [[str(a) for a in range(self.M)] for _ in range(self.N)]

In [ ]:
ebasej.actions(slf)

[['0', '1', '2', '3'], ['0', '1', '2', '3'], ['0', '1', '2', '3']]

In [ ]:
#| export
@patch
def states(self:ebasej):
        """Default state set representation `state_s`."""
        return [str(s) for s in range(self.Z)]

In [ ]:
ebasej.states(slf)

['0', '1']

In [ ]:
#| export
@patch
def observations(self:ebasej):
        """Default observation set representations `obs_io`."""
        if hasattr(self, 'defaultObsTensUsed'):
            return [[str(o) for o in self.states()] for _ in range(self.N)]
        else:
            return [[str(o) for o in range(self.Q)] for _ in range(self.N)]


In [ ]:
ebasej.observations(slf)

[['0', '1', '2', '3', '4'],
 ['0', '1', '2', '3', '4'],
 ['0', '1', '2', '3', '4']]

In [ ]:
#| export

@patch
def id(self:ebasej):
    """
    Returns id string of environment
    """
    return f"{self.__class__.__name__}"

@patch
def __str__(self:ebasej): 
    return self.id()

@patch
def __repr__(self:ebasej): 
    return self.id()

## Interactive use
Environments can also be used interactivly, e.g., with iterative learning algorithms. For this purpose we provide the [OpenAI Gym `step` Interface](https://github.com/openai/gym#api).

In [ ]:
#| export

@patch
def step(self:ebasej, jA: Iterable) -> tuple:
    """
    Iterate the environment one step forward.
    
    Parameters:
        jA: Iterable representing joint actions (one action per agent, as indices)
    
    Returns:
        A tuple: (observations, rewards, done, info)
    """
    # Choose a next state according to transition tensor T
    # Build an index: current state followed by the joint action indices.
    idx = (self.state,) + tuple(jA)
    tps = self.T[idx].astype(float)
    
    # Update random key and choose the next state using jax.random.choice.
    self.rng, subkey = random.split(self.rng)
    next_state = int(random.choice(subkey, a=jnp.arange(tps.shape[0]), p=tps))
    
    # Obtain the rewards:
    # The reward tensor is indexed as: (agent, current state, joint actions..., next_state)
    rewards = self.R[(slice(None), self.state) + tuple(jA) + (next_state,)]
    
    # Advance the state
    self.state = next_state
    
    # Obtain observations from the new state.
    obs = self.observation()
    
    # Determine if the new state is a final state (episode done)
    done = bool(self.state in jnp.where(self.F == 1)[0])
    
    # Package info (for debugging or analysis)
    info = {'state': self.state}
    
    return obs, rewards.astype(float), done, info

In [ ]:
#| export
@patch
def observation(self:ebasej) -> jnp.ndarray:
    """
    Possibly random observation for each agent from the current state.
    
    Returns:
        A JAX array of observations (one per agent).
    """
    obs_list = []
    for i in range(self.N):
        # Get observation probabilities for agent i at current state.
        ops = self.O[i, self.state]
        self.rng, subkey = random.split(self.rng)
        obs_i = int(random.choice(subkey, a=jnp.arange(ops.shape[0]), p=ops))
        obs_list.append(obs_i)
    return jnp.array(obs_list)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()